# 01 — Détection d'humour : le problème, binaire contre typologie mécanique

**Étape 2 / carnet d'intro de [#20301](https://github.com/jsboige/CoursIA/issues/20301)** (corpus d'humour annoté par la série [HumorDetection](README.md)). Ce carnet pose le terrain : qu'est-ce que *détecter* l'humour, qu'est-ce que le *typer*, et pourquoi la série a besoin des deux. Les exemples sont construits pour le cours ; aucun corpus externe n'est téléchargé ici — le paysage des datasets vient au carnet 02.

## 1. Deux questions différentes : « est-ce de l'humour ? » et « par quel mécanisme ? »

La **détection binaire** demande si un texte est humoristique. La **typologie mécanique** demande *ce qui fait rire* — la forme du mécanisme comique. La série retient trois formes (couche 1 du banc ICT-35) :

- **pun** — jeu de mots : le mécanisme est lexical, deux sens coexistent dans le même mot ;
- **sociale** — blague sociale intemporelle : le mécanisme est la reconnaissance d'une situation partagée ;
- **topical** — référence culturelle ou métier d'époque : le mécanisme est l'ancrage dans un contexte daté, et la blague vieillit avec lui.

Une deuxième couche existe (taxonomie Argumentum, branche Humour PK 219, 17 feuilles) : le carnet 03 construit la correspondance entre les deux couches. Ici, seule la couche 1 est manipulée.

In [1]:
# Dix exemples courts, construits pour le cours (pas des extraits de corpus).
# est_humour : étiquette binaire. forme : étiquette typologique (None pour les témoins).
exemples = {
    "H1": {"texte": "Quel est le comble pour un électricien ? De ne pas être au courant.", "est_humour": True, "forme": "pun"},
    "H2": {"texte": "Les mathématiciens ne meurent pas vraiment : ils perdent leurs fonctions.", "est_humour": True, "forme": "pun"},
    "H3": {"texte": "Que dit un escargot quand il croise une limace ? — T'as oublié ta coquille !", "est_humour": True, "forme": "sociale"},
    "H4": {"texte": "Pourquoi les plongeurs plongent-ils toujours en arrière ? Parce que sinon, ils tombent dans le bateau.", "est_humour": True, "forme": "sociale"},
    "H5": {"texte": "Un homme entre dans un café. Ploum.", "est_humour": True, "forme": "sociale"},
    "H6": {"texte": "En 2026, le seul fichier qui se remplit tout seul, c'est le journal de la CI.", "est_humour": True, "forme": "topical"},
    "H7": {"texte": "Un modèle de langage entre dans un bar et commande un prompt. Le barman répond : désolé, je suis en rate limit.", "est_humour": True, "forme": "topical"},
    "T1": {"texte": "La réunion de suivi est reportée à jeudi 14 h, salle 3B.", "est_humour": False, "forme": None},
    "T2": {"texte": "L'eau bout à 100 degrés Celsius au niveau de la mer.", "est_humour": False, "forme": None},
    "T3": {"texte": "Le formulaire C-43 doit être rempli en deux exemplaires.", "est_humour": False, "forme": None},
}

humoristiques = sum(1 for e in exemples.values() if e['est_humour'])
print(len(exemples), 'exemples :', humoristiques, 'humoristiques,', len(exemples) - humoristiques, 'témoins non humoristiques')
for cle, e in exemples.items():
    forme = e['forme'] or 'témoin'
    print(cle + ': [' + forme + '] ' + e['texte'][:72])

10 exemples : 7 humoristiques, 3 témoins non humoristiques
H1: [pun] Quel est le comble pour un électricien ? De ne pas être au courant.
H2: [pun] Les mathématiciens ne meurent pas vraiment : ils perdent leurs fonctions
H3: [sociale] Que dit un escargot quand il croise une limace ? — T'as oublié ta coquil
H4: [sociale] Pourquoi les plongeurs plongent-ils toujours en arrière ? Parce que sino
H5: [sociale] Un homme entre dans un café. Ploum.
H6: [topical] En 2026, le seul fichier qui se remplit tout seul, c'est le journal de l
H7: [topical] Un modèle de langage entre dans un bar et commande un prompt. Le barman 
T1: [témoin] La réunion de suivi est reportée à jeudi 14 h, salle 3B.
T2: [témoin] L'eau bout à 100 degrés Celsius au niveau de la mer.
T3: [témoin] Le formulaire C-43 doit être rempli en deux exemplaires.


### Lecture des dix exemples

H1 et H2 sont des **pun** : le ressort est dans un mot (« au courant », « fonctions ») qui porte deux sens. H3-H5 sont des **sociale** : le ressort est une situation partagée (l'escargot et la limace, le plongeur, l'entrée au café) qui survivrait à une traduction. H6-H7 sont des **topical**, mais sur deux ancrages de nature différente : H6 sur une date (« En 2026, ... journal de CI »), H7 sur un vocabulaire métier (« barman », « prompt », « rate limit »). Retirez l'ancrage d'époque et la blague tombe à plat — c'est le test opérationnel de la forme. T1-T3 sont des **témoins** : mêmes longueurs, même registre factuel, aucune intention comique. Un détecteur binaire rend le même verdict sur H1 et H7 ; la typologie les sépare — et c'est tout l'enjeu du corpus que la série va annoter.

### Exercice 1 — Étiqueter avant de lire

**Contexte.** Les étiquettes vivent dans `exemples`. Avant de les lire, portez votre propre jugement sur les dix textes : binaire d'abord, forme ensuite. C'est le geste que le protocole d'annotation (carnet 05) industrialisera avec deux annotateurs LLM.

- **Étape 1 :** pour chaque clé, décider `est_humour` (vrai/faux) sans regarder la valeur rangée.
- **Étape 2 :** pour les humoristiques, décider la forme parmi `pun` / `sociale` / `topical`.
- **Étape 3 :** comparer avec les étiquettes de référence et calculer votre accord binaire puis typologique.

**Indice :** un des sept humoristiques est discutable en forme (H5) ; le débat fait partie de l'exercice, et c'est exactement ce que le kappa du protocole mesurera.

In [2]:
# Exercice 1 à compléter : votre étiquetage, puis l'accord avec la référence.
# Étape 1 : remplir mon_etiquetage = {cle: (bool, forme_ou_None), ...} pour les dix clés.
# Étape 2 : comparer champ à champ avec exemples[k]['est_humour'] / exemples[k]['forme'].
# Étape 3 : afficher l'accord binaire et l'accord typologique (2 accords sur 10 items).
mon_etiquetage = None  # TODO étudiant : votre dict {cle: (est_humour, forme)}
print('Exercice a completer' if mon_etiquetage is None else 'voir la suite')

Exercice a completer


In [3]:
# Répartition de référence : la typologie voit ce que le binaire ne voit pas.
from collections import Counter

par_forme = Counter(e['forme'] for e in exemples.values() if e['est_humour'])
print('Humoristiques par forme :', dict(par_forme))
print('Témoins non humoristiques :', sum(1 for e in exemples.values() if not e['est_humour']))
print('Le binaire dit seulement :', sum(1 for e in exemples.values() if e['est_humour']), 'contre',
      sum(1 for e in exemples.values() if not e['est_humour']))

Humoristiques par forme : {'pun': 2, 'sociale': 3, 'topical': 2}
Témoins non humoristiques : 3
Le binaire dit seulement : 7 contre 3


### Lecture du résultat

La ligne « par forme » porte une information que le total binaire n'a pas : la **composition** du corpus. Un détecteur entraîné sur un corpus déséquilibré (beaucoup de pun, peu de topical) apprend la forme majoritaire, pas l'humour. C'est la raison pour laquelle le corpus final de la série sera **équilibré par forme** (équilibrage par exposant α, carnet 06) — et pourquoi le carnet 05 mesurera l'accord **par forme**, pas seulement global : un kappa global peut masquer une forme sur laquelle les annotateurs ne sont jamais d'accord.

## 2. Le problème ColBERT, posé sur nos exemples

Le dataset **ColBERT Humor** (200 000 textes courts EN, licence MIT) pose le problème en binaire, avec des **contrôles non-humoristiques intégrés** — la propriété qui l'a retenu à l'étape 1. Avant d'y toucher (carnet 02, après re-vérification de licence et archivage bibliographique), on peut poser le problème ici : une baseline lexicale suffit-elle sur nos dix exemples ? Celle ci-dessous marque un texte comme humoristique s'il contient un déclencheur de forme interrogative ou d'exclamation — deux marqueurs lexicaux fréquents des blagues courtes.

In [4]:
# Baseline lexicale maison : humoristique si la question ou l'exclamation est présente.
DECLENCHEURS = ('?', '!')

def baseline_lexicale(texte):
    return any(m in texte for m in DECLENCHEURS)

justes = 0
for cle, e in exemples.items():
    pred = baseline_lexicale(e['texte'])
    reel = 'humour' if e['est_humour'] else 'témoin'
    ok = pred == e['est_humour']
    justes += ok
    print(cle + ': prédit=' + ('humour' if pred else 'sérieux') + ' / réel=' + reel + ' -> ' + ('OK' if ok else 'ÉCHEC'))
print('Exactitude de la baseline lexicale : ' + str(justes) + '/' + str(len(exemples)))

H1: prédit=humour / réel=humour -> OK
H2: prédit=sérieux / réel=humour -> ÉCHEC
H3: prédit=humour / réel=humour -> OK
H4: prédit=humour / réel=humour -> OK
H5: prédit=sérieux / réel=humour -> ÉCHEC
H6: prédit=sérieux / réel=humour -> ÉCHEC
H7: prédit=sérieux / réel=humour -> ÉCHEC
T1: prédit=sérieux / réel=témoin -> OK
T2: prédit=sérieux / réel=témoin -> OK
T3: prédit=sérieux / réel=témoin -> OK
Exactitude de la baseline lexicale : 6/10


### Lecture de la baseline

La baseline réussit sur les blagues à question ou à exclamation — mais **échoue sur tout le reste** : H2 (« Les mathématiciens ne meurent pas vraiment : ils perdent leurs fonctions. »), H5 (« Un homme entre dans un café. Ploum. ») et H6 n'ont aucun déclencheur lexical, et H2 et H5 sont pourtant des blagues canoniques. Inversement, un témoin bureaucratique bien ponctué pourrait la piéger. La leçon n'est pas « il faut plus de marqueurs » — c'est que **le mécanisme comique n'est pas lexical** : il est sémantique (deux sens, situation partagée, ancrage culturel). C'est exactement pourquoi ColBERT apprend un classifieur sur 200 000 textes plutôt qu'une liste de marqueurs, et pourquoi la typologie (pun / sociale / topical) décrit le **mécanisme** là où la baseline ne voit que la **surface**.

### Exercice 2 — Compter les formes, proprement

**Contexte.** Le comptage de référence ci-dessus utilise `Counter`. On veut maintenant la table croisée binaire × forme, témoins inclus, et la part de chaque forme parmi les humoristiques — c'est le tableau que le rapport du protocole (carnet 05) produira sur des milliers de paires.

- **Étape 1 :** construire un dict imbriqué `{forme: {'humour': n, 'temoin': n}}` couvrant les trois formes et le témoin.
- **Étape 2 :** en dériver la part de chaque forme parmi les humoristiques (trois fractions).
- **Étape 3 :** vérifier qu'aucune forme ne dépasse la moitié du corpus humoristique — c'est la garde d'équilibre que le carnet 06 appliquera à grande échelle.

In [5]:
# Exercice 2 à compléter : table croisée forme × binaire + parts par forme.
# Étape 1 : table = {forme: {'humour': 0, 'temoin': 0}} pour pun/sociale/topical et le témoin.
# Étape 2 : part = humour_de_la_forme / total_humoristiques, pour les trois formes.
# Étape 3 : assert aucune part > 0.5, avec un message explicite sinon.
table_croisee = None  # TODO étudiant : votre dict imbriqué
print('Exercice a completer' if table_croisee is None else 'voir la suite')

Exercice a completer


### Exercice 3 — Choisir le pilote

**Contexte.** L'étape 1 de #20301 a retenu trois candidats sérieux : **ColBERT Humor** (200 000 textes courts EN, MIT, témoins intégrés, binaire), **HAHA 2021** (espagnol, compétition de détection d'humour, évaluation externe) et **Jester** (blagues longues avec notes, source d'intensité). Le pilote doit servir l'annotation à deux annotateurs du carnet 05.

- **Étape 1 :** écrire une fonction `choisir_pilote(candidats)` qui prend un dict `{candidat: {'témoins': bool, 'licence': str, 'taille': int}}` et retourne le nom du pilote selon la règle : **témoins intégrés d'abord, licence établie ensuite, taille enfin**.
- **Étape 2 :** l'appliquer aux trois candidats avec les valeurs de l'étape 1 (ColBERT : témoins oui, MIT, 200 000, EN ; HAHA : témoins oui, licence compétition, ~24 000, ES ; Jester : témoins non, usage recherche, ~100 blagues notées, EN).
- **Étape 3 :** dire en commentaire quelle ligne de la règle a décidé, et ce qui changerait si le volet FR devenait prioritaire (réponse attendue : aucun pilote externe ne fournit le FR — c'est le protocole enseignant qui le fera).

In [6]:
# Exercice 3 à compléter : la règle de décision du pilote.
# Étape 1 : def choisir_pilote(candidats) -> str, règle : témoins puis licence puis taille.
# Étape 2 : l'appliquer aux trois candidats de l'étape 1 et afficher le pilote retenu.
# Étape 3 : commenter la ligne décisive et le cas du volet FR.
pilote = None  # TODO étudiant : nom du candidat retenu par votre règle
print('Exercice a completer' if pilote is None else pilote)

Exercice a completer


## 3. Ce que cette série construira — et ce que ce carnet ne montre pas

- **Carnet 02** : le paysage des datasets **en accès réel** (téléchargement effectif après re-vérification de licence et archivage bibliographique), l'écart EN/FR chiffré.
- **Carnet 03** : l'articulation entre la couche 1 (formes) et la couche 2 (taxonomie Argumentum, 17 feuilles d'Humour + 4 feuilles de la vertu « Humour de bon goût »).
- **Carnets 04-06** : le corpus contrôlé, le protocole à deux annotateurs LLM **de familles distinctes** (Qwen côté hub vLLM, annotateur B non-Qwen via claudish), l'accord kappa 0,6 en **signal** (passe de désambiguïsation documentée, jamais de rejet silencieux), puis le builder cartésien.

**Ce que ce carnet ne montre pas** : aucune mesure sur un vrai détecteur (ColBERT n'est pas téléchargé), aucun accord inter-annotateurs (dix exemples ne mesurent rien), aucune donnée externe. Il pose le terrain ; les mesures vivent dans les carnets suivants, chacune committée avec son protocole.

## Conclusion

Détecter l'humour en binaire et le typer par son mécanisme sont deux tâches différentes, et la seconde est celle qui vaut un corpus : elle seule dit *pourquoi* un texte fait rire, et donc pourquoi un détecteur échoue sur certains textes et pas d'autres. La baseline lexicale de ce carnet échoue exactement là où la théorie des formes le prédit (H5, H6 : mécanisme non lexical). La suite de la série industrialise le geste d'étiquetage de l'exercice 1 — deux annotateurs, accord mesuré par forme, reclassage aller-retour — et produit le corpus annoté que la strate ICT consommera.